---
# 🤖 Interpretación de Keyness con un LLM (Python + Hugging Face)
---

Este notebook carga la tabla de *keyness* exportada desde R (`tstat_key_movies.csv`, generada en `4_Keyness _ Dispersión Textual.ipynb`) y usa un modelo de lenguaje **gratuito y de código abierto** (vía Hugging Face `transformers`) para ayudar a interpretar los resultados.

Flujo:
1. Cargar el CSV con `pandas`.
2. Separar las palabras más asociadas a reseñas **positivas** vs. **negativas** según el signo de `chi2`.
3. Construir un prompt compacto con las palabras clave.
4. Enviarlo a un modelo instruccional open-source para obtener una interpretación cualitativa.

> 📌 **Sobre el costo:** este notebook no requiere API key ni tarjeta de crédito. El modelo se descarga una vez desde Hugging Face y corre localmente (o en la GPU gratuita de Colab). Es más lento y menos sofisticado que un modelo comercial como Claude u GPT-4, pero suficiente para explorar el flujo de trabajo sin costo alguno.
>
> Si estás en **Google Colab**, activa GPU en `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4)` antes de correr las celdas — en CPU funciona, pero es notablemente más lento.

In [2]:
import pandas as pd

df = pd.read_csv("tstat_key_movies.csv")
df.head()

,feature,chi2,p,n_target,n_reference
0,life,114.328553,0.000000e+00,983,487
1,jackie,85.279582,0.000000e+00,210,47
2,truman,79.375662,0.000000e+00,121,11
3,great,69.314320,1.110223e-16,744,396
4,mulan,67.322652,2.220446e-16,79,1


### 🔍 Explorando la tabla

Columnas esperadas (salida de `textstat_keyness()` en quanteda):

- `feature`: la palabra.
- `chi2`: estadístico de keyness. **Signo positivo** → asociada al grupo *target* (reseñas `pos`). **Signo negativo** → asociada al grupo de referencia (reseñas `neg`).
- `p`: valor p asociado.
- `n_target`, `n_reference`: frecuencias observadas en cada grupo.

In [3]:
N = 30

top_pos = df.sort_values("chi2", ascending=False).head(N)
top_neg = df.sort_values("chi2", ascending=True).head(N)

print("Top palabras asociadas a reseñas POSITIVAS:")
display(top_pos[["feature", "chi2", "n_target", "n_reference"]])

print("\nTop palabras asociadas a reseñas NEGATIVAS:")
display(top_neg[["feature", "chi2", "n_target", "n_reference"]])

Top palabras asociadas a reseñas POSITIVAS:


,feature,chi2,n_target,n_reference
0,life,114.328553,983,487
1,jackie,85.279582,210,47
2,truman,79.375662,121,11
3,great,69.314320,744,396
4,mulan,67.322652,79,1
5,war,63.269628,259,90
6,world,53.075391,624,341
7,excellent,52.836041,146,37
8,shrek,52.397014,59,0
9,flynt,52.237035,62,1



Top palabras asociadas a reseñas NEGATIVAS:


,feature,chi2,n_target,n_reference
47931,bad,-405.793450,357,1021
47930,movie,-179.459432,2393,3053
47929,worst,-169.849954,49,259
47928,stupid,-124.731294,45,207
47927,boring,-123.182695,52,218
47926,plot,-103.832606,575,876
47925,supposed,-95.597145,83,239
47924,godzilla,-92.932738,14,116
47923,waste,-81.087910,22,121
47922,ridiculous,-76.973157,22,117


---
## 💬 Interpretación con un LLM open-source (Hugging Face)

Usamos [`Qwen2.5-3B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct), un modelo instruccional pequeño, gratuito y sin restricciones de acceso (no requiere solicitar permiso ni iniciar sesión en Hugging Face para descargarlo).

Construimos un prompt con las listas de palabras y le pedimos que identifique patrones temáticos o estilísticos que distinguen a las reseñas positivas de las negativas — el mismo tipo de análisis que haríamos con un modelo comercial, pero sin costo.

In [10]:
%pip install -q transformers accelerate torch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch
from transformers import pipeline

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

generator = pipeline(
    "text-generation",
    model=MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

/Users/fernandodiaz/dev-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.73it/s]
Device set to use mps


In [5]:
pos_words = ", ".join(top_pos["feature"].tolist())
neg_words = ", ".join(top_neg["feature"].tolist())

prompt = f"""Estas son las palabras más distintivas (análisis de keyness, chi-cuadrado) del corpus de
críticas de cine de Pang & Lee (2004), separadas en dos grupos:

Palabras asociadas a reseñas POSITIVAS:
{pos_words}

Palabras asociadas a reseñas NEGATIVAS:
{neg_words}

Analiza estas listas e identifica:
1. Patrones temáticos o de vocabulario que distinguen a cada grupo.
2. Si hay palabras que te resulten sorprendentes o difíciles de explicar solo por el sentimiento.
3. Una hipótesis breve sobre por qué estas palabras (y no otras más obviamente positivas/negativas)
   emergen como las más distintivas según el estadístico de keyness.

Responde en español, de forma concisa (máximo ~300 palabras)."""

messages = [{"role": "user", "content": prompt}]

output = generator(
    messages,
    max_new_tokens=500,
    temperature=0.7,
    do_sample=True,
)

respuesta = output[0]["generated_text"][-1]["content"]
print(respuesta)

Las palabras distintivas en ambas listas ofrecen patrones temáticos y de vocabulario significativos.

Palabras asociadas a reseñas positivas destacan temas como la experiencia humana ("life", "family"), personajes icónicos ("jackie", "truman", "mulan", "damon", "cameron"), grandes éxitos ("great", "excellent", "wonderful", "outstanding", "perfect", "memorable", "fictitious"), y películas icónicas ("titanic", "toy", "lebowski", "jedi"). Estas palabras sugieren una reseña que se centra en experiencias humanas, personajes memorables y grandes éxitos cinematográficos.

Por otro lado, las palabras asociadas a reseñas negativas se concentran en aspectos negativos de las películas, incluyendo malos resultados ("bad", "worst", "stupid", "boring", "mess"), problemas con los guiones ("plot", "script"), y elementos que se consideran irrelevantes o inútiles ("waste", "nothing", "lame", "terrible"). Esto sugiere una crítica enfocada en la calidad del producto, los defectos del guion y la falta de c

---
### 📄 Exportar la respuesta a PDF

Guardamos `respuesta` (el texto generado por el modelo) en un PDF legible — útil para leerlo completo fuera del recuadro de salida de VS Code, o para compartirlo.

In [6]:
%pip install -q fpdf2

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [14]:
from fpdf import FPDF

# Los LLM suelen generar comillas tipográficas, guiones largos, etc. que no
# existen en Latin-1 (la codificación que usan las fuentes básicas de fpdf2).
# Los reemplazamos por sus equivalentes ASCII antes de escribir el PDF.
_REEMPLAZOS = {
    "—": "-",   # — em dash
    "–": "-",   # – en dash
    "‘": "'",   # ‘
    "’": "'",   # ’
    "“": '"',   # “
    "”": '"',   # ”
    "…": "...", # …
    " ": " ",   # espacio de no separación
    "•": "-",   # • bullet
}


def sanitize_for_pdf(text: str) -> str:
    for original, reemplazo in _REEMPLAZOS.items():
        text = text.replace(original, reemplazo)
    # Red de seguridad: cualquier otro carácter fuera de Latin-1 (p. ej. emojis
    # o símbolos no previstos arriba) se reemplaza por "?" en vez de romper el PDF.
    return text.encode("latin-1", errors="replace").decode("latin-1")


pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", size=12)
pdf.set_title("Interpretacion de Keyness (LLM)")

pdf.set_font("Helvetica", style="B", size=14)
pdf.multi_cell(0, 10, sanitize_for_pdf("Interpretación de Keyness - Críticas de cine (Pang & Lee, 2004)"))
pdf.ln(4)

pdf.set_font("Helvetica", size=11)
pdf.multi_cell(0, 7, sanitize_for_pdf(respuesta))

output_path = "interpretacion_keyness.pdf"
pdf.output(output_path)
print(f"PDF guardado en: {output_path}")

PDF guardado en: interpretacion_keyness.pdf


> ✅ A partir de aquí puedes iterar: cambiar `N`, probar otro modelo open-source de Hugging Face (por ejemplo `microsoft/Phi-3-mini-4k-instruct`), o combinar esto con los resultados de `kwic()` (contexto de uso) que generaste en el notebook de R para enriquecer el análisis.
>
> **Nota sobre calidad:** un modelo de 3B parámetros es mucho más limitado que modelos comerciales grandes (Claude, GPT-4, etc.). Sus interpretaciones pueden ser más genéricas o menos precisas — esto es parte de la discusión pedagógica: ¿qué se gana y qué se pierde al usar un modelo pequeño y gratuito vs. uno comercial de pago?